# Neural Network for Stress Prediction

Surrogate model to predict stress from wing geometry parameters (`W1`, `W2`, `R`, `t`), trained on FEA simulation data. Architecture and hyperparameters are selected via cross-validation and compared against the GPR baseline.

## Imports

In [11]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

In [2]:
# Set a fixed random state for reproducibility across all random operations
random_state = 42

# Set the path to the dataset
data_path = "../data/TrainingData.csv"

## Data Pre-Processing

### Load & Inspect Dataset

In [3]:
# Load the dataset, including missing headers and assigning column names
df = pd.read_csv(data_path, header=None, names=['W1', 'W2', 'R', 't', 'stress'])

### Train / Test Split

Hold out 20% as a test set — never touched during architecture search or training.

In [ ]:
# split into features and target
X = df.drop('stress', axis=1)
y = df['stress']

# split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

## Neural Network Model

### Architecture Search

Define candidate architectures (depth, width, activation function) to compare via K-Fold cross-validation. Metrics used for selection: CV MAPE.

In [17]:
# create PyTorch datasets and dataloaders
train_dataset = TensorDataset(torch.tensor(X_train.values, dtype=torch.float32), torch.tensor(y_train.values, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test.values, dtype=torch.float32), torch.tensor(y_test.values, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Inspect the shape of the first batch of data
for X, y in test_loader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([10, 4])
Shape of y: torch.Size([10]) torch.float32


In [15]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [6]:
# define a dictionary to store depth, width and activation function combinations to assess in the grid search
nn_configs = {
    'config_1': {'hidden_layer_sizes': (50,), 'activation': 'relu'},
    'config_2': {'hidden_layer_sizes': (100,), 'activation': 'relu'},
    'config_3': {'hidden_layer_sizes': (50, 50), 'activation': 'relu'},
    'config_4': {'hidden_layer_sizes': (100, 100), 'activation': 'relu'},
    'config_5': {'hidden_layer_sizes': (50,), 'activation': 'tanh'},
    'config_6': {'hidden_layer_sizes': (100,), 'activation': 'tanh'},
    'config_7': {'hidden_layer_sizes': (50, 50), 'activation': 'tanh'},
    'config_8': {'hidden_layer_sizes': (100, 100), 'activation': 'tanh'},
}

In [ ]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

### Training Configuration

Set optimiser, loss function, learning rate scheduler, early stopping patience, and batch size.

In [8]:
# loop through the configurations, train a model for each and evaluate its performance on the test set, storing the results in a dictionary
results = {}
for config_name, config_params in nn_configs.items():
    # Create the MLPRegressor with the specified parameters and a fixed random state
    model = MLPRegressor(hidden_layer_sizes=config_params['hidden_layer_sizes'],
                         activation=config_params['activation'],
                         random_state=random_state)
    
    # Train the model on the training data
    model.fit(X_train, y_train)
    
    # Evaluate the model on the test set and store the results
    test_score = model.score(X_test, y_test)
    results[config_name] = {'test_score': test_score}
# Identify the best configuration based on test score
best_config = max(results, key=lambda k: results[k]['test_score'])
print(f"Best configuration: {best_config} with test score: {results[best_config]['test_score']}")

/opt/anaconda3/envs/data-driven/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/data-driven/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/data-driven/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/data-driven/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Best configuration: config_4 with test score: -0.0034164064102795155


/opt/anaconda3/envs/data-driven/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/data-driven/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


### Model Training & Evaluation

Cross-validate each candidate architecture on the training set, then evaluate the best on the held-out test set. Metrics: CV MAPE (architecture selection), Test RMSE, MAE, MAPE, training time, and per-sample prediction time.

### Visualisations

#### Loss Curves

Training and validation loss over epochs for the best architecture. Verify convergence and check for overfitting.

#### Architecture Comparison: Predicted vs True

Scatter plot of predicted vs true stress for each candidate architecture. The dashed red line is the ideal fit.

#### Residual Analysis

Residuals (predicted − true) plotted against true stress and against each input feature to diagnose systematic bias.

#### Prediction Surface (W1 vs W2)

3D surface of predicted stress as `W1` and `W2` vary, with `R` and `t` held at their training-set means. Black points show training observations projected onto the surface.

### Model Selection & Export

Save the best-performing model and its scaler for use in the optimisation script.

## Comparison with GPR Baseline

Side-by-side comparison of the best NN against the best GPR model (Matern 2.5 kernel) on the same held-out test set. Metrics: Test RMSE, MAE, MAPE, training time, prediction time.